# TokenVector.Inference vs PyTorch (Colab T4 GPU)

GPT-2 124M, seq1024, seed 7, xen kÃ¡ÂºÂ½ x3, median.

ChÃ¡ÂºÂ¡y theo thÃ¡Â»Â© tÃ¡Â»Â±. KhÃƒÂ´ng cÃ¡ÂºÂ§n sÃ¡Â»Â­a gÃƒÂ¬ nÃ¡ÂºÂ¿u mÃ¡Â»Âi thÃ¡Â»Â© xanh.

**ChÃƒÂº ÃƒÂ½:**

- GPU bridge (ATen) segfault rc=139 duoc fix trong repo Release commit `8929830` (dev/host dung tgt.copy_(t)); ky vong [7/9] rc=0, loss_val = 17.106686 (phien #22).
- Neu [7/9] van fail rc=139: chay [9/9 diag-gdb] - backtrace moi chi ra loi con lai.
- Bench GPU [7/9] KHONG abort; Bench native [6/9] luon chay duoc.
- Ket qua ghi lai vao `benchmark/BENCHMARKS.md` kem phien (luat #1 `docs/TRACKED_PLAN.md`).


In [ ]:
# 1/9 GPU + torch check
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda.is_available():', torch.cuda.is_available())
import subprocess
_r = subprocess.run('nvcc --version', shell=True, capture_output=True, text=True)
print('nvcc rc =', _r.returncode)
print((_r.stdout or '') + (_r.stderr or ''))
_r2 = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True, capture_output=True, text=True)
print('compute cap:', _r2.stdout.strip())


In [ ]:
# 2/9 Install .NET 10 SDK
import urllib.request
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
urls = [
    'https://dot.net/v1/dotnet-install.sh',
    'https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh',
]
for u in urls:
    try:
        with urllib.request.urlopen(u, timeout=30) as r:
            open('/tmp/dotnet-install.sh', 'wb').write(r.read())
        print('Downloaded .NET install script from:', u)
        break
    except Exception as e:
        print('FAIL', u, e)
else:
    raise RuntimeError('KhÃƒÂ´ng tÃ¡ÂºÂ£i Ã„â€˜Ã†Â°Ã¡Â»Â£c dotnet-install.sh')
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ['PATH'] = '/opt/dotnet:' + os.environ['PATH']


In [ ]:
# 3/9 Clone repos + LD_LIBRARY_PATH (torch CUDA) + verify binary
import os, glob, subprocess
for repo in ['TokenVector', 'TokenVector.Inference']:
    p = subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/{}.git 2>&1 || (cd {} && git pull --depth 1 2>&1)'.format(repo, repo), shell=True, capture_output=True, text=True)
    print(repo, '->', 'ok' if p.returncode == 0 else 'fail: ' + p.stderr[-300:])
print('Inference HEAD:', subprocess.check_output('git -C TokenVector.Inference rev-parse --short HEAD', shell=True).decode().strip())
print('TokenVector HEAD:', subprocess.check_output('git -C TokenVector rev-parse --short HEAD', shell=True).decode().strip())
import hashlib
b = hashlib.sha256(open('TokenVector.Inference/demo/bseq.dll','rb').read()).hexdigest()
print('demo/bseq.dll sha256:', b)
src = open('TokenVector.Inference/tv/tests/tv.training.gpt2_ref124.tkv', encoding='utf-8').read()
print('A4a fallback qkv fix present:', 'CON CAN qkv' in src)
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
torch_dir  = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/torch')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
print('torch     :', torch_dir)
ld = []
if cublas_dir: ld.append(cublas_dir)
if cudart_dir: ld.append(cudart_dir)
if torch_dir:  ld.append(torch_dir + '/lib')
ld.append('/usr/lib64-nvidia')
os.environ['LD_LIBRARY_PATH'] = ':'.join(ld)
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# 4/9 Build tkv_simd_native.so + tkv_step124.so (Ã„â€˜Ã¡Â»â€¢i tÃƒÂªn .dll Ã„â€˜Ã¡Â»Æ’ khÃ¡Â»â€ºp pinvoke)
import subprocess
r = subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c -lm -lpthread -ldl -o tkv_simd_native.dll', shell=True, capture_output=True, text=True)
print('native build rc =', r.returncode)
if r.returncode != 0: print(r.stderr[-3000:])
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/demo/tkv_simd_native.dll', shell=True)
print('copy rc =', r.returncode)
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/tkv_simd_native.dll', shell=True)
print('copy2 rc =', r.returncode)
r = subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll', shell=True, capture_output=True, text=True)
print('step124 rc =', r.returncode)
if r.returncode != 0: print(r.stderr[-2000:])
r = subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_step124.dll /content/TokenVector.Inference/demo/tkv_step124.dll', shell=True)
print('copy3 rc =', r.returncode)
import ctypes
try:
    lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
    lib.TkvCUDA_Available.restype = ctypes.c_int
    print('TkvCUDA_Available() =', lib.TkvCUDA_Available())
except Exception as e:
    print('load native failed:', e)


In [ ]:
# 5/9 Build tkv_torch_fused.so (ATen bridge)
import subprocess, os, sys
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
incs = [torch_dir + '/include', torch_dir + '/include/torch/csrc/api/include']
libdir = torch_dir + '/lib'
cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
       'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
       '-I' + incs[0] + ' -I' + incs[1] + ' -L' + libdir + ' '
       '-Wl,-rpath,' + libdir + ' '
       '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda')
p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
print('build rc =', p.returncode)
if p.returncode != 0:
    print('STDERR:', p.stderr[:6000])
print('STDOUT tail:', (p.stdout or '')[-1500:])
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = libdir + ':' + os.environ.get('LD_LIBRARY_PATH', '')
import ctypes
try:
    lib = ctypes.CDLL(so_path)
    print('tkv_torch_fused.so: ctypes load OK')
except Exception as e:
    print('ctypes load fail:', e)


In [ ]:
# 7/9 Bench GPU bridge (TKV_TORCH_GPU=1) Ã¢â‚¬â€ crash-tolerant, KHÃƒâ€NG abort
import re, statistics as st, subprocess
DOT = '/opt/dotnet/dotnet'
def tkv_gpu():
    out = subprocess.run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 ' + DOT + ' demo/bseq.dll', shell=True, capture_output=True, text=True, timeout=1800)
    return out.returncode, (out.stdout or '') + '\n' + (out.stderr or '')
res = []
for r in range(3):
    rc, o = tkv_gpu()
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+) loss_val=([\d.]+)', o)
    if rc == 0 and m:
        d = dict(zip(('fwd','loss','bwd','opt','copyback','TONG','loss_val'), map(float, m.groups())))
        res.append(d)
        print('%d: TONG=%5.0f  fwd=%5.0f bwd=%5.0f | loss_val=%.6f' % (r+1, d['TONG'], d['fwd'], d['bwd'], d['loss_val']))
    else:
        print('%d: GPU BRIDGE FAIL rc=%d' % (r+1, rc))
        print(o[-1200:])
if res:
    m = lambda k: st.median(x[k] for x in res)
    print()
    print('--- MEDIAN (GPU bridge x%d) ---' % len(res))
    print('.tkv TONG     : %.1f ms' % m('TONG'))
    print('fwd / bwd     : %.0f / %.0f ms' % (m('fwd'), m('bwd')))
    print('opt / copyback: %.0f / %.0f ms' % (m('opt'), m('copyback')))
    print('loss_val      : %.6f' % res[0]['loss_val'])
else:
    print()
    print('GPU BRIDGE KONG CHAY DUOC tren phien nay (segfault).')
    print('Native ben [6/9] van la so dang tin cay. GPU nhay ngau nhien theo phien Colab.')
    print('Neu can GPU: chay lai lan nua / doi phien, hoac kiem tra TkvTorchAttentionBwd trong [8/9].')

# GPU state ngay sau bench (tu chan doan throttle/degrade khi so sanh giua cac phien)
import subprocess as _sp
_q = 'nvidia-smi --query-gpu=name,clocks.sm,power.draw,temperature.gpu,utilization.gpu --format=csv'
print(_sp.run(_q, shell=True, capture_output=True, text=True).stdout)


In [ ]:
# 7B/9 Full-step GPU resident (1 native call/step) - flag CFG dong 4 = 1

import subprocess

DOT = '/opt/dotnet/dotnet'

open('TokenVector.Inference/tv/tests/_seq_cfg.txt','w').write('1024\n2\n1\n1\n')

# GPU monitor: sample nvidia-smi trong luc chay (peak RAM/GPU)
import threading, time
_peak = {'mem': 0, 'util': 0}
_done = threading.Event()
def _watch_gpu():
    while not _done.is_set():
        try:
            _r = subprocess.run('nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits', shell=True, capture_output=True, text=True, timeout=5)
            _v = (_r.stdout or '').strip().splitlines()
            if _v:
                _pp = [x.strip() for x in _v[0].split(',')]
                _peak['mem'] = max(_peak['mem'], int(float(_pp[0])))
                _peak['util'] = max(_peak['util'], int(float(_pp[1])))
        except Exception:
            pass
        time.sleep(0.5)
_w = threading.Thread(target=_watch_gpu, daemon=True)
_w.start()
p = subprocess.run('cd TokenVector.Inference && TKV_FULL_GPU=1 ' + DOT + ' demo/bseq.dll', shell=True, capture_output=True, text=True, timeout=1800)

print('rc=', p.returncode)

print((p.stdout or '')[-1500:])

print('ERR:', (p.stderr or '')[-800:])
_done.set()
_w.join(timeout=2)
print('GPU peak: mem=' + str(_peak['mem']) + ' MiB, util=' + str(_peak['util']) + ' %')


In [ ]:
# 11/9 LN bench: LN fwd+bwd micro vs torch native
import torch, torch.nn.functional as F, time
torch.manual_seed(0)
x = torch.randn(1024, 768, requires_grad=True)
g = torch.randn(768, requires_grad=True); b = torch.randn(768, requires_grad=True)
gy = torch.randn(1024, 768)
def timeit(fn, n=20):
    for _ in range(5): fn()
    t0 = time.perf_counter()
    for _ in range(n): fn()
    return (time.perf_counter() - t0) / n * 1000.0
print('torch LN fwd :', timeit(lambda: F.layer_norm(x, (768,), g, b)), 'ms')
def ln_bwd():
    y = F.layer_norm(x, (768,), g, b, 1e-6)
    y.backward(gy)
    x.grad = None; g.grad = None; b.grad = None

try:
    print('torch LN bwd :', timeit(ln_bwd), 'ms')
except Exception as e:
    print('ln bwd bench skip:', type(e).__name__, e)
print()
print('.tkv LN timing xem o [8/10 PROF] Ã¢â‚¬â€ LN_FWD/LN_BWD ~ ns/call se cho biet bridge co chay')
